<table>
  <tr>
    <td><div align="left"><font size="30">Cart-pole system</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Modeling the classic cart-pole (inverted pendulum) system using *bdsim***

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


# Theory

A cart of mass $M$ rolls along a horizontal track and a pole is pivoted on top of it.  The position of the cart along the track is $x$.  The angle of the pole $\theta$ is measured from the upright vertical and increases clockwise, so $\theta > 0$ when the pole leans in the direction of positive $x$.  A horizontal force $F$ is applied to the cart in the direction of positive $x$.  The force is the system input, and the aim is to keep the pole balanced upright.  This is a classic problem in control education because the upright position is an equilibrium, but an unstable one, like a broom balanced on your hand: a small lean grows exponentially unless the cart moves to get back underneath the pole.

We assume that:

* the pole is a point mass $m$ on the end of a rigid massless rod of length $l$ (a "lumped mass"),
* the pivot is frictionless, and the cart experiences viscous friction along the track with coefficient $b$,
* the motion is planar, and
* the actuator is perfect: the force $F$ is whatever we command, applied directly to the cart, with no limit.

The equations of motion can be derived using any number of techniques such as Lagrangian or Newton-Euler dynamics, and are given in many places, for example [Underactuated Robotics, Chapter 3, Russ Tedrake, MIT](https://underactuated.csail.mit.edu/acrobot.html#cart_pole).  Tedrake measures the pole angle from the downward vertical, $\theta_T = \pi - \theta$, and in terms of our upright angle $\theta$ they are
$$
\begin{align}
\ddot{x} &= \frac{f + m \sin \theta \left( l \dot{\theta}^2 - g \cos \theta \right)}{D} \\
\ddot{\theta} &= \frac{-f \cos \theta - m l \dot{\theta}^2 \cos \theta \sin \theta + (M + m) g \sin \theta}{l D}
\end{align}
$$
where $f = F - b \dot{x}$ is the nett force on the cart (applied minus friction) and $D = M + m \sin^2 \theta$ comes from inverting the mass matrix.  The key step in building a block diagram model is to isolate the second derivatives of the state variables, which we then feed into a chain of integrators.

For small $\theta$ equation (2) behaves like $\ddot{\theta} \approx \frac{(M+m) g}{M l} \theta$, which has a positive sign: the further the pole leans, the faster it accelerates away from vertical.

As a sanity check we will compute the total energy of the system.  The kinetic energy is
$$
T = \frac{M + m}{2} \dot{x}^2 + \frac{m l^2 \dot{\theta}^2}{2} + m l \dot{x} \dot{\theta} \cos \theta \qquad (3)
$$
and the potential energy, which we take to be zero with the pole hanging straight down, is
$$
U = m g l (1 + \cos \theta) \qquad (4)
$$
The total energy of the system is $E = T + U$.  The pivot and the pole itself have no friction, so energy enters only through the applied force and leaves only through friction on the cart
$$
\dot{E} = F \dot{x} - b \dot{x}^2 \qquad (5)
$$
and once the force is removed the energy can only decrease.

## Parameters

The parameter values used in this notebook are

| Quantity | Value |
| --- | --- |
| Mass of cart ($M$) | 1.0 kg |
| Mass of pole ($m$) | 0.1 kg |
| Length of pole ($l$) | 0.5 m |
| Viscous friction coefficient of cart ($b$) | 10 N s/m |
| Gravitational acceleration ($g$) | 9.81 m/s² |

# Implementation with *bdsim*

We start with the standard imports, construct a simulator object, and use that to construct
a block diagram with all the `bdsim` blocks included as methods.  The cart and pole move
quickly, so we set `animation_rate=10`: the scope and animation are refreshed 10 times per
second of simulated time.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True, animation_rate=10)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

Next up, we define the parameters of the system

In [ ]:
M = 1.0   # mass of cart (kg)
m = 0.1   # mass of pole (kg)
l = 0.5   # length of pole (m)
g = 9.81  # gravity (m/s^2)
b = 10    # viscous friction coefficient of cart (N s/m)

The system can be modeled as two chains of integrators.  The first integrates cart acceleration $\ddot{x}$ to velocity $\dot{x}$ to position $x$.  The initial position and velocity are zero.

In [ ]:
x_dot = bd.INTEGRATOR(name="x_dot")
x = x_dot >> bd.INTEGRATOR(name="x")

Note that we are using the right-shift operator to implicitly **connect** the output of the `x_dot` block to the input of the next integrator, and the result of that expression is assigned to `x`.

The other chain integrates the pole's angular acceleration $\ddot{\theta}$ to angular velocity $\dot{\theta}$ to angle $\theta$.  The initial angle and velocity are zero, the pole pointing straight up.

In [ ]:
theta_dot = bd.INTEGRATOR(name="theta_dot")
theta = theta_dot >> bd.INTEGRATOR(name="theta")

We use a pair of function blocks to compute the sine and cosine of the pole angle

In [ ]:
cos_theta = theta >> bd.FUNCTION(math.cos, name="cos(theta)")
sin_theta = theta >> bd.FUNCTION(math.sin, name="sin(theta)")

The system starts in an unstable equilibrium with all states zero: the cart and pole not moving, and the pole pointing upwards.
We apply a force pulse of 1 N to the cart for $t \in [1, 1.2]$ s to disturb it.

The applied force is the pulse minus a state-feedback term $\mathbf{K}x$.  We include the feedback now, but with a gain of zero, so
the system is open loop.  Once we have designed a controller we will set the gain, without changing anything else in the model.  The
`STATE` block outputs a vector of the states we name, in the order we name them, and the gain block is given the name `K` so that we can find it later.

In [ ]:
state = bd.STATE(["x_dot", "x", "theta_dot", "theta"])  # the state vector, in this order
Kgain = bd.GAIN(np.zeros((1, 4)), premul=True, name="K", inputs=[state])  # state feedback gain, zero for now

F = bd.PIECEWISE((0, 0), (1, 1), (1.2, 0), name="disturbance") - Kgain  # force on the cart (N)

Next we compute the common term $D$ and the nett force on the cart $f$

In [ ]:
D = M + m * sin_theta**2  # from the inverse of the mass matrix
f = F - b * x_dot  # nett force on the cart (input minus friction)

and then the cart acceleration (1) and the pole angular acceleration (2), which are the inputs to the two chains of integrators

In [ ]:
x_dot[0] = (
    f + m * sin_theta * (l * theta_dot**2 - g * cos_theta)
) / D  # cart acceleration (m/s^2)
theta_dot[0] = (
    -f * cos_theta
    - m * l * theta_dot**2 * cos_theta * sin_theta
    + (M + m) * g * sin_theta
) / (l * D)  # pole angular acceleration (rad/s^2)

As a cross check we compute the total energy of the system $E$ (3)-(4) and connect that to a scope along with the force, the cart position and the pole angle.
After the force pulse, which adds a little energy, we should see the total energy of the system decrease monotonically (5).  The energy of the swinging pole is dissipated by the friction of the cart.

In [ ]:
KE = (
    1 / 2 * (M + m) * x_dot**2
    + 1 / 2 * m * l**2 * theta_dot**2
    + m * l * x_dot * theta_dot * cos_theta
)  # kinetic energy (J), note the coupling term between the cart and pole
PE = m * g * l * (1 + cos_theta)  # potential energy (J), zero with the pole hanging down
E = KE + PE  # total energy (J)

# display the system state evolution on a scope and animation
scope = bd.SCOPE(
    inputs=[F, x, theta, E],
    labels=("F (N)", "x (m)", r"$\theta$ (rad)", "E (J)"),
    loc="lower right",
)

To include a graphical animation of our cart and pole system we need to create two functions: graphics setup,
and graphics update.  We will use simple Matplotlib
patch graphics – this kind of code is tedious to write, and it can be outsourced to a coding agent.  The graphics are drawn
to scale, so the pole is $l$ long and the point mass at its tip is shown as a circle.

In [ ]:
import matplotlib.patches as patches
import matplotlib.transforms as transforms


def cartpole_init(self, fig, ax, l=l):
    # l is the length of the pole
    # 1. Setup the figure and axis
    ax.set_xlim(-0.9, 0.9)
    ax.set_ylim(-0.6, 0.8)
    ax.set_aspect("equal")
    ax.grid(True, linestyle="--")

    # Draw the horizontal track
    ax.axhline(0, color="black", lw=1.5)

    # 2. Define geometry constraints
    cart_w, cart_h = 0.3, 0.15
    pole_w = 0.02
    bob_r = 0.04

    # Pivot point relative to the cart: top-middle
    self.pivot_y_rel = cart_h

    # 3. Create the patches at the origin (0,0)
    # Cart anchor is bottom-left, so we center it horizontally at X=0
    cart = patches.Rectangle(
        (-cart_w / 2, 0),
        cart_w,
        cart_h,
        linewidth=1.5,
        edgecolor="blue",
        facecolor="lightblue",
    )

    # Pole anchor is bottom-left. Center it horizontally at X=0, starting at the pivot
    pole = patches.Rectangle(
        (-pole_w / 2, 0),
        pole_w,
        l,
        linewidth=1.5,
        edgecolor="red",
        facecolor="salmon",
    )

    # The lumped mass at the end of the pole
    bob = patches.Circle((0, l), bob_r, linewidth=1.5, edgecolor="red", facecolor="salmon")

    ax.add_patch(cart)
    ax.add_patch(pole)
    ax.add_patch(bob)

    self.cart = cart
    self.pole = pole
    self.bob = bob


def cartpole_update(self, t, u):
    cart_x = u[0]
    pole_angle = u[1]
    ax = self.ax

    # --- Cart Transformation ---
    # 1. Shift the cart horizontally by cart_x
    # 2. Transform from model to display coordinates (+ ax.transData)
    cart_trans = transforms.Affine2D().translate(cart_x, 0) + ax.transData
    self.cart.set_transform(cart_trans)

    # --- Pole Transformation ---
    # 1. Rotate the pole around its own bottom-center (0,0 in its local space), clockwise for positive angle
    # 2. Translate it to match the cart's current absolute pivot position
    # 3. Transform from model to display coordinates (+ ax.transData)
    pole_trans = (
        transforms.Affine2D().rotate(-pole_angle).translate(cart_x, self.pivot_y_rel)
    ) + ax.transData

    self.pole.set_transform(pole_trans)
    self.bob.set_transform(pole_trans)

These two functions are passed to an `ANIMATION` block driven by the outputs of the `x` and `theta` blocks.  These inputs
are the elements of `u` passed to `cartpole_update`.

In [ ]:
anim = bd.ANIMATION(
    init=cartpole_init, update=cartpole_update, nin=2, inputs=[x, theta]
)

Our system is built, we will compile it, and display a summary

In [ ]:
bd.compile()  # check the diagram
print(bd)

# bd.report_summary()

The model comprises 50 blocks, but the majority were created implicitly (block names are prefixed by `_`) by the arithmetic expressions. A full list of all blocks in the model can be displayed by uncommenting the line above, and rerunning the cell. To create all these blocks and connect them would be both tedious and error prone.  The power of *bdsim* is that it lets
you move effortlessly between expression in terms of blocks or using Python expressions.

We simulate the open-loop system for 20 seconds.  The force pulse accelerates the cart to the right, which leaves the pole behind
and tips it to the left (negative $\theta$).  The upright equilibrium is unstable, so the pole falls over, passing through the horizontal
at about 2.2 s.  It swings down past the hanging position and back up to within 40° of the vertical on the other side, reaching an angle
of $-5.58$ rad ($-320°$) at about 2.9 s, and then continues to swing, with decreasing amplitude, about the hanging position $\theta = -\pi$.
The cart hardly moves, no more than about 5 cm from where it started, because the friction coefficient is large.

The energy scope confirms the sanity check.  The system starts with $E = 2 m g l = 0.981$ J, the potential energy of the upright pole.
The pulse adds about 0.004 J, and from then on the energy falls monotonically as the cart friction removes the energy of the swinging pole, reaching 0.13 J by the end of the run.

In [ ]:
out = sim.run(bd, T=20, watch=[F, x, theta])  # simulate for 20s
print(out)

# Linearization

We can linearize this non-linear system around an equilibrium operating point, here the upright pole with the cart at rest.
An operating point is a state together with the input that holds it there, so we give both to `linearize`: all states zero,
and no force.  The input is the output of the `disturbance` block, and the outputs are the cart position and the pole angle.

In [ ]:
lin = bd.linearize(x=[0, 0, 0, 0], u=[0], inputs=["disturbance"], outputs=[x, theta], full=True)
print(lin)

The states are in the order $(\dot{x}, x, \dot{\theta}, \theta)$, the order of the blocks as they were added to the diagram.  For small angles
$\sin \theta \approx \theta$, $\cos \theta \approx 1$ and $D \approx M$, and equations (1) and (2) become

$$
\begin{aligned}
\ddot{x} &= \frac{1}{M}\left( f - m g \theta \right) \\
\ddot{\theta} &= \frac{1}{M l}\left( -f + (M + m) g \theta \right)
\end{aligned}
$$

where $f = F - b \dot{x}$, and so
$$
\mathbf{A} = \begin{pmatrix}
-b/M & 0 & 0 & -m g/M \\
1 & 0 & 0 & 0 \\
b/(M l) & 0 & 0 & (M+m) g/(M l) \\
0 & 0 & 1 & 0
\end{pmatrix}, \qquad
\mathbf{B} = \begin{pmatrix} 1/M \\ 0 \\ -1/(M l) \\ 0 \end{pmatrix}
$$
which we can compare with the numerical result

In [ ]:
A_hand = np.array([
    [-b / M,       0, 0, -m * g / M],
    [1,            0, 0, 0],
    [b / (M * l),  0, 0, (M + m) * g / (M * l)],
    [0,            0, 1, 0],
])
B_hand = np.array([[1 / M], [0], [-1 / (M * l)], [0]])
print(np.allclose(lin.A, A_hand), np.allclose(lin.B, B_hand))
print(lin.poles)

There is one pole in the right-half plane, at $s = +4.50$ rad/s.  This is the falling pole, which grows by a factor of $e$ every
$1/4.50 = 0.22$ s, so a controller has very little time to react.  The pole at $s = 0$ is the integrator that turns cart velocity into
position: nothing restores a displaced cart to its original position.  The other two are stable.

The matrix $\mathbf{B}$ shows that a positive force accelerates the cart to the right, by $1/M = 1$ m/s² per newton, but accelerates the pole to the left, by $-1/(M l) = -2$ rad/s² per newton, since the pole is left behind.

# Control

## Control design

We will design a linear controller using the Python Control package, which is not a dependency of `bdsim`

```
$ pip install control
```

and build an LQ state-feedback controller to hold the pole upright and the cart at the origin, despite a disturbance force.
We start by exporting the *bdsim* linearization as a Python Control state-space object, with its `ss()` method.  The state names are copied across,
and this is a reminder of the order of the states.

In [ ]:
import control as ct

sys = lin.ss()  # export as a Python Control state-space object, with the state names
print(sys.state_labels)

To design an LQ regulator we need to define the $Q$ (penalizes state errors) and $R$ (penalizes control effort) matrices.
$Q$ must be positive semi-definite and $R$ must be positive definite -- both are typically chosen to be diagonal.
We apply the same penalty to the cart position and the pole angle, the two quantities we care about, and none to the velocities.
The control effort is the force on the cart, in newtons.  The weights are given in the order of the states listed above.

In [ ]:
Q = np.diag([0, 10, 0, 10])  # weights for [x_dot, x, theta_dot, theta]
R = np.array([[1]])          # weight for control input, the force F

To find the optimal state-feedback $F = -\mathbf{K}x$ we need to solve the algebraic Riccati equation

In [ ]:
K, S, poles_cl = ct.lqr(sys, Q, R)
print(K)

where `K` is the optimal state-feedback gain matrix, `S` is the solution to the algebraic Riccati equation, and `poles_cl`
are the eigenvalues (poles) of the closed-loop system ($A - BK$).

To solve the algebraic Riccati equation the package uses an efficient Fortran-based package `slicot` if it is available (via the Python wrapper [slycot](https://github.com/python-control/Slycot)) otherwise it falls back to a pure-Python solution implemented in SciPy.

The gain matrix $K$ has one row per input and one column per state.  The gain on the pole angle is large and the control force is
$F = 21.8 \dot{x} + 3.2 x + 15.5 \dot{\theta} + 69.9 \theta$: when the pole leans, push the cart in the direction it is leaning, to get underneath it.
The gain on the cart position has the "wrong" sign, $F$ increases with $x$.  This is a property of the cart-pole: to move the cart to the left
you must first push it to the right, to tilt the pole to the left, and only then push to the left to catch it.

## Converting open-loop model to closed-loop

We have already built the feedback path into the model, with a gain of zero.  To close the loop we just set the value of the gain block

In [ ]:
Kgain.K = K  # set the gain block to the computed LQR gain

As a check, we can linearize the closed-loop diagram.  Its poles should be the closed-loop poles that the design predicted

In [ ]:
lin_cl = bd.linearize(x=[0, 0, 0, 0], u=[0], inputs=["disturbance"], outputs=[x, theta], full=True)
print(lin_cl.poles)
print(np.allclose(lin_cl.poles, np.sort(poles_cl.real)))

The unstable pole has been moved into the left-half plane, and all four poles are real.  The slowest, at $s = -0.32$ rad/s, is a time constant of about 3 s, and
governs how quickly the cart drifts back to the origin after the pole has been caught.

We simulate the closed-loop system for 20 seconds, with the same force pulse as before.  We can observe:
* the pole no longer falls over.  The pole angle never exceeds 0.012 rad (0.7°) and returns to vertical,
* the cart is displaced by at most 1.6 cm and is back within 2 mm of the origin in about 9 s, the time taken by the slowest pole,
* the total force on the cart peaks at only 1.2 N, not much more than the 1 N pulse.  The controller's response to the pulse is small because the pole barely moves,
* the energy stays within 0.002 J of its upright value of 0.981 J, since the pole never leaves the vertical.

In [ ]:
out = sim.run(bd, T=20, watch=[F, x, theta])  # simulate for 20s

## Exercises for the reader

* Plot the control force.  We have not limited the force at all.
  * The linear controller recovers from a pulse 40 times larger (40 N, giving a peak pole angle of 27°). Is that believable?
  * Limit the force to $\pm 5$ N with a `CLIP` block, and find how large a pulse the controller can recover from.
* Add a setpoint for the cart position, a step of 0.2 m, so that the control law becomes $F = -\mathbf{K}(x - x_d)$.
  * You will see the cart move briefly in the wrong direction before moving to the new position.  How large is this effect?
* The slowest pole is set by the weight on the cart position in $Q$.  Reduce that weight and watch the slowest pole move closer to the origin and the cart take longer to return.
* We assumed a perfect actuator:
  * Add a model of a motor with a first-order lag between the controller and the cart, and investigate how the response changes as the lag increases.
* Investigate model error: change the pole length, the friction coefficient $b$ or the mass of the pole in the plant but not in the controller.
* The energy $E$ is a good sign of how far the pole is from upright.  The controller above only works near the vertical:
  * Start the pole hanging down ($\theta(0) = \pi$), and design an energy-based *swing-up* controller that adds energy until $E = 2 m g l$.
  * Switch to the LQ controller when the pole is close enough to the vertical.
* Improve the control performance:
  * adjust the $Q$ and $R$ matrices
  * try a different control strategy, such as:
    * PID control (you will need two loops, one for the pole angle and one for the cart position),
    * model predictive control (MPC),
    * pole placement.